In [12]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Stats
from scipy import stats
from scipy.stats import (
    chi2_contingency, kruskal, mannwhitneyu, f_oneway,
    pointbiserialr, spearmanr, kendalltau, pearsonr,
)

# sklearn
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Multicollinearity (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display + plot defaults
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)

In [15]:
# Load the dataset
DATA_PATH = "Data/The Hartford Data/UConn_2026_data(in).csv"
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
df.head()

Shape: (300000, 44)


,AUTO_POL_TRANS_ID,RISK_ST_ABBR,POL_EFF_DT,POL_EXP_DT,POL_NEW_RENEW_CD,POL_TTL_BILL_PREM_AMT,RISK_GRP,RISK_FLAG,CLTV_LONGVTY_GRP_CD,AARP_MEMB_YR_CNT,AARP_MEMB_IND,FULL_PAY_DISC_IND,POL_BI_OCCUR_LMT_CD,CLEAN_DIRTY_DESC,DRVR_TLMTC_PGM_TYP_CD,RENEW_YRS,POL_MULTI_SNGL_VEH_CD,POL_DRVR_TLMTC_ENROL_IND,ORIG_INTRNT_RATE_QTE_IND,PPV_CNT,PCARR_DESC,AVG_VEH_PREM,PCARR_YR_CNT,PCARR_CALC_YR_CNT,POL_FORM_CD,POL_RATE_DRVR_MIN_AGE,POL_RATE_DRVR_MAX_AGE,HH_COMP,ADV_QTE_DAY_CNT,AQD_GRP,YOUTHFUL_IND,BILL_PYMNT_METH_DESC,BILL_PYMENT_FREQ_DESC,AVG_VEHICLE_MILEAGE,MKT_CHNNL_NM,MKTG_MEDIA_GRP_DESC,MKTG_NON_BRAND_SEARCH_FLAG,ORIG_QTE_CNTCT_METH_CD,E_SIGN_IND,ECNSNT_BEGINING_OF_POL_TERM_IND,PLCY_MIN_CAR_BASE_PRICE,PLCY_MAX_CAR_BASE_PRICE,PLCY_AVG_CAR_BASE_PRICE,POL_ACCT_CR_IND
0,1064231476,CO,5/12/2025,11/12/2025,R,864.0,6,NaN,PL16,1,Y,1.0,500.0,Dirty,M,1.0,S,N,N,1,Others,864.00,2.0,2.9,0,63,63,SC1D,50.0,14-60,N,CreditOrDebitCard,PayInFull,10000,Search,AARP Branded,Other than Non-Branded Search,P,0,1,23245.0,23245.0,23245.0,0
1,1018298329,WI,3/2/2023,9/2/2023,N,287.0,9,NaN,PL22,1,Y,1.0,300.0,Dirty,X,0.0,S,N,Y,1,Others,287.00,1.0,0.9,H*,24,59,SCMT1D,25.0,14-60,Y,CreditOrDebitCard,RepetitiveFullPay,8000,Online,HIG.com,Other than Non-Branded Search,I,1,1,27170.0,27170.0,27170.0,0
2,1057068242,FL,9/24/2024,3/24/2025,R,980.0,5,NaN,NaN,8,Y,0.0,20.0,Dirty,X,2.5,S,Y,N,1,Others,980.00,2.0,2.8,H*,58,58,SC1D,82.0,75+,N,BankAccount,MonthlyAutoPay,11000,Direct Mail,Winback,Other than Non-Branded Search,P,1,1,21045.0,21045.0,21045.0,0
3,1044629296,FL,9/24/2024,3/24/2025,R,2079.0,8,NaN,NaN,12,Y,0.0,500.0,Clean,X,1.0,M,N,N,2,Geico,1039.50,5.0,6.2,H*,64,71,MC C=D,27.0,14-60,N,BankAccount,MonthlyAutoPay,10000,NaN,NaN,Other than Non-Branded Search,P,1,0,25095.0,29705.0,27400.0,0
4,1053922228,SC,7/13/2024,1/13/2025,R,3925.0,1,NaN,PL24,1,Y,0.0,50.0,Dirty,X,0.5,M,Y,Y,3,Others,1308.33,5.0,3.9,0,39,61,MCMCTD,15.0,14-60,N,BankAccount,MonthlyAutoPay,12453,Search,Non Branded,Search Non Branded,I,1,1,22970.0,32190.0,27392.0,0


Swagat Sainju

### Meera Chanda - EDA Findings

### Bundlers vs Non-Bundlers

In [37]:
# Target class balance: bundlers vs non-bundlers
target = "POL_ACCT_CR_IND"

target_counts = df[target].value_counts(dropna=False)
target_percent = df[target].value_counts(normalize=True, dropna=False) * 100

target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_percent.round(2)
})

target_summary

,count,percent
POL_ACCT_CR_IND,,
0,196716,65.57
1,103284,34.43


In our data set of 300,000 records, roughly 65.6% (197,000) didn't bundle, and only 34.4% (103,000) did. The classes are not balanced 50/50.

### Missing values

In [39]:
# Missing values summary
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

missing_summary.head(15)

,missing_count,missing_percent
RISK_FLAG,294128,98.04
CLTV_LONGVTY_GRP_CD,88773,29.59
MKTG_MEDIA_GRP_DESC,46249,15.42
MKT_CHNNL_NM,46249,15.42
AQD_GRP,17265,5.76
ADV_QTE_DAY_CNT,16099,5.37
PCARR_CALC_YR_CNT,9932,3.31
PLCY_AVG_CAR_BASE_PRICE,8087,2.70
PLCY_MAX_CAR_BASE_PRICE,8087,2.70
PLCY_MIN_CAR_BASE_PRICE,8087,2.70


- The largest missingness issue is `RISK_FLAG`, which is missing **98.04%** of values.
- `CLTV_LONGVTY_GRP_CD` is missing **29.59%**, which is important because customer longevity may be related to bundling behavior. 
- Marketing/channel fields also have moderate missingness, including `MKTG_MEDIA_GRP_DESC` and `MKT_CHNNL_NM` at **15.42%** each.
- Most columns have low missingness, but highly missing variables should be reviewed before modeling.

### Comparing missingness by target group

In [48]:
# Compare missingness by target group
missing_by_target = {}

for col in df.columns:
    if col != target:
        missing_by_target[col] = df.groupby(target)[col].apply(lambda x: x.isna().mean() * 100)

missing_by_target_df = pd.DataFrame(missing_by_target).T
missing_by_target_df = missing_by_target_df.round(2)

missing_by_target_df.head(20)

POL_ACCT_CR_IND,0,1
AUTO_POL_TRANS_ID,0.00,0.00
RISK_ST_ABBR,0.00,0.00
POL_EFF_DT,0.00,0.00
POL_EXP_DT,0.00,0.00
POL_NEW_RENEW_CD,0.00,0.00
POL_TTL_BILL_PREM_AMT,0.00,0.00
RISK_GRP,0.00,0.00
RISK_FLAG,97.47,99.13
CLTV_LONGVTY_GRP_CD,39.55,10.62
AARP_MEMB_YR_CNT,0.00,0.00


- `RISK_FLAG` is highly missing for both groups, but slightly higher among bundlers (**99.13%**) than non-bundlers (**97.47%**). 
- `CLTV_LONGVTY_GRP_CD` shows a larger difference: it is missing for **39.55%** of non-bundlers but only **10.62%** of bundlers, which suggests that missingness in customer longevity can be informative for predicting bundling.

### Top categorical predictors

In [49]:
# Top categorical predictors using Cramér’s V

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    
    if confusion_matrix.shape[0] < 2 or confusion_matrix.shape[1] < 2:
        return np.nan
    
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    
    return np.sqrt((chi2 / n) / min(k - 1, r - 1))

categorical_cols = df.select_dtypes(include=["object", "category"]).columns
categorical_cols = [col for col in categorical_cols if col != target]

cramers_results = []

for col in categorical_cols:
    v = cramers_v(df[col].fillna("Missing"), df[target])
    cramers_results.append({
        "feature": col,
        "cramers_v": round(v, 3)
    })

cramers_df = pd.DataFrame(cramers_results).sort_values("cramers_v", ascending=False)

cramers_df.head(15)

,feature,cramers_v
12,POL_FORM_CD,0.999
4,CLTV_LONGVTY_GRP_CD,0.503
0,RISK_ST_ABBR,0.413
16,BILL_PYMNT_METH_DESC,0.216
11,PCARR_DESC,0.158
1,POL_EXP_DT,0.145
7,DRVR_TLMTC_PGM_TYP_CD,0.117
19,MKTG_MEDIA_GRP_DESC,0.114
13,HH_COMP,0.105
8,POL_MULTI_SNGL_VEH_CD,0.092


- `POL_FORM_CD` is too predictive because its Cramér’s V is 0.999, which may be data leakage. We can exclude this variable. 
- `CLTV_LONGVTY_GRP_CD` is the strongest useful predictor of bundling, with a Cramér’s V of 0.503. This suggests that customer longevity is strongly related to whether a customer bundles auto and home insurance.
-  Other important predictors include `RISK_ST_ABBR`, `BILL_PYMNT_METH_DESC`, and `PCARR_DESC`.

### Numeric feature correlations with target

In [50]:
# Numeric feature correlations with target
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
numeric_cols = [col for col in numeric_cols if col != target]

numeric_corrs = []

for col in numeric_cols:
    corr = df[[col, target]].corr().iloc[0, 1]
    numeric_corrs.append({
        "feature": col,
        "correlation_with_target": corr
    })

numeric_corrs_df = pd.DataFrame(numeric_corrs)
numeric_corrs_df["absolute_correlation"] = numeric_corrs_df["correlation_with_target"].abs()

numeric_corrs_df = numeric_corrs_df.sort_values("absolute_correlation", ascending=False)

numeric_corrs_df.head(15)

,feature,correlation_with_target,absolute_correlation
5,POL_BI_OCCUR_LMT_CD,0.243984,0.243984
8,AVG_VEH_PREM,-0.163165,0.163165
2,RISK_GRP,0.157275,0.157275
15,E_SIGN_IND,-0.150491,0.150491
3,AARP_MEMB_YR_CNT,0.138809,0.138809
0,AUTO_POL_TRANS_ID,0.107053,0.107053
1,POL_TTL_BILL_PREM_AMT,-0.100828,0.100828
12,POL_RATE_DRVR_MAX_AGE,0.094542,0.094542
4,FULL_PAY_DISC_IND,0.078662,0.078662
7,PPV_CNT,0.071928,0.071928


- Numeric features have some predictive value, but they are weaker than the strongest categorical variables. 
- Our model should include numeric features like coverage limit, average vehicle premium, risk group, and AARP membership years, but the most important signals may come from categorical/customer profile variables such as customer longevity, state, billing method, and prior carrier.

### Suspicious leakage check

In [51]:
# Check suspicious leakage-related columns
possible_leakage_cols = [
    "POL_FORM_CD",
    "E_SIGN_IND",
    "ECNSNT_BEGINING_OF_POL_TERM_IND"
]

for col in possible_leakage_cols:
    if col in df.columns:
        print(f"\n{col}")
        print(pd.crosstab(df[col].fillna("Missing"), df[target], normalize="index").round(3))


POL_FORM_CD
POL_ACCT_CR_IND    0    1
POL_FORM_CD              
0                1.0  0.0
H*               1.0  0.0
HO3              0.0  1.0
HO4              0.0  1.0
HO6              0.0  1.0

E_SIGN_IND
POL_ACCT_CR_IND      0      1
E_SIGN_IND                   
0                0.576  0.424
1                0.720  0.280

ECNSNT_BEGINING_OF_POL_TERM_IND
POL_ACCT_CR_IND                      0      1
ECNSNT_BEGINING_OF_POL_TERM_IND              
0                                0.662  0.338
1                                0.652  0.348


- `POL_FORM_CD` is highly suspicious because it appears to almost perfectly separate bundlers from non-bundlers. It has a Cramér's V ≈ 1.0 This could suggest the variable may contain information that would notbe available at the time of prediction, so it should be excluded.

- `E_SIGN_IND` and `ECNSNT_BEGINING_OF_POL_TERM_IND` show weaker differences and should hold off until we can confirm if they were recorded before the bundling decision.

### Bundler Profile by Important Categorical Features

In [22]:
# Bundling rate by selected categorical variables
important_cat_cols = [
    "CLTV_LONGVTY_GRP_CD",
    "RISK_ST_ABBR",
    "POL_BI_OCCUR_LMT_CD",
    "BILL_PYMNT_METH_DESC",
    "RISK_GRP",
    "PCARR_DESC"
]

for col in important_cat_cols:
    if col in df.columns:
        print(f"\nBundling rate by {col}")
        display(
            df.groupby(col)[target]
            .agg(["count", "mean"])
            .rename(columns={"mean": "bundling_rate"})
            .sort_values("bundling_rate", ascending=False)
            .head(10)
        )


Bundling rate by CLTV_LONGVTY_GRP_CD


,count,bundling_rate
CLTV_LONGVTY_GRP_CD,,
PL04,584,0.984589
PL05,5640,0.978546
PL06,7322,0.921333
PL07,8737,0.751059
PL12,9580,0.658559
PL13,10836,0.644241
PL14,11944,0.629186
PL11,8271,0.611655
PL08,9045,0.567938



Bundling rate by RISK_ST_ABBR


,count,bundling_rate
RISK_ST_ABBR,,
IN,8630,0.625492
MI,8258,0.615645
IA,1582,0.607459
ND,85,0.552941
ID,761,0.551905
OH,14454,0.547738
WI,6229,0.546958
IL,22411,0.539958
MN,1340,0.526866



Bundling rate by POL_BI_OCCUR_LMT_CD


...

,count,bundling_rate
BILL_PYMNT_METH_DESC,,
BankAccount,193851,0.418512
CreditOrDebitCard,102534,0.201728



Bundling rate by RISK_GRP


,count,bundling_rate
RISK_GRP,,
10,29998,0.465564
9,29985,0.404669
8,30133,0.389573
6,30120,0.381839
7,29923,0.381212
5,29556,0.367844
4,30009,0.332833
3,30120,0.286886
2,30153,0.246808



Bundling rate by PCARR_DESC


,count,bundling_rate
PCARR_DESC,,
Farmers,6324,0.459994
Liberty Mutual,11367,0.438198
Others,121691,0.400112
StateFarm,27606,0.369847
AllState,36807,0.362458
USAA,5721,0.319525
Progressive,49084,0.247963
Geico,41400,0.221111


- `CLTV_LONGVTY_GRP_CD` shows a strong pattern, with some longevity groups having very high bundling rates, suggesting that customer tenure/relationship history is important. 
- Bundling differs by state, billing method, bodily injury coverage limit, risk group, and prior carrier.
- Customers using `BankAccount` payments have a much higher bundling rate than those using `CreditOrDebitCard`, and higher coverage/risk categories also show stronger bundling behavior. 

### Numeric Profile of Bundlers vs. Non-Bundlers

In [23]:
# Compare numeric averages for bundlers vs non-bundlers
important_num_cols = [
    "AVG_VEH_PREM",
    "POL_TTL_BILL_PREM_AMT",
    "AVG_CAR_PRICE",
    "AARP_MEMB_YR_CNT",
    "CARR_CALC_YR_CNT"
]

existing_num_cols = [col for col in important_num_cols if col in df.columns]

df.groupby(target)[existing_num_cols].mean().round(2)

,AVG_VEH_PREM,POL_TTL_BILL_PREM_AMT,AARP_MEMB_YR_CNT
POL_ACCT_CR_IND,,,
0,806.01,1107.40,10.94
1,654.88,956.37,14.05


 - Non-bundlers have a higher average vehicle premium (**$806.01**) and total bill premium (**$1107.40**) compared to bundlers, who average **$654.88** and **$956.37**. 
 - However, bundlers have a higher average AARP membership duration (**14.05 years**) compared to non-bundlers (**10.94 years**). 
 - This suggests that longer AARP membership may be associated with a greater likelihood of bundling.

### Bundling rate by year

In [52]:
# Bundling rate by year
df["POL_EFF_DT"] = pd.to_datetime(df["POL_EFF_DT"], errors="coerce")
df["policy_year"] = df["POL_EFF_DT"].dt.year

yearly_bundle_rate = (
    df.groupby("policy_year")[target]
    .agg(["count", "mean"])
    .rename(columns={"mean": "bundling_rate"})
)

yearly_bundle_rate["bundling_rate"] = (yearly_bundle_rate["bundling_rate"] * 100).round(2)

yearly_bundle_rate

,count,bundling_rate
policy_year,,
2022,28282,26.14
2023,70241,29.23
2024,107490,34.15
2025,93987,41.13


- Bundling rates increased each year, rising from **26.14% in 2022** to **41.13% in 2025**, whichsuggests that home-auto bundling became more common over time. 
- For the project, year/time may be an important factor, and we should consider using a time-based train/test split or including policy year as a feature.